# LCentral AI Lead Conversion Agent (prototype)

Built to the *Project Brief - LCentral AI Lead Conversion Agent v1.2*. The agent works **alongside the Centre Success Team (CST)**; it does not chat with parents.

> **The Agent captures, understands and recommends. The system reminds and highlights. The CST decides and acts.** (Brief §3)

| Brief requirement | Where it lives in this notebook |
|---|---|
| §5A Improve lead capture: free-text note → structured lead, CST reviews and confirms | §8 AI capture, app tab **1. Capture interaction** |
| §5B Next Best Action: Owner + Stage + Need + NBA + Due date; flag rather than guess | §5 Action Standards engine, §9 operating loop |
| §5C Make sure the action happens: due today, overdue, escalations, high-intent, no NBA, assessments/trials without outcomes, stale | §10 Action board, app tab **2. Today's board** |
| §5C Operating loop: Interaction → Understand → Capture → NBA → Act → Outcome → Update → NBA | §9, app tab **3. Lead record** |
| §6 LCentral Action Standards applied, not invented | §5 (deterministic rules, every action cites the rule it came from) |
| §7–8 Re-engagement drafts from an approved content library, CST reviews before sending | §11, app tab **4. Draft message** |
| §9 Human-in-the-loop and CST feedback signals (edit rate, overrides) | §11, §12 |
| §14 Success measures | §12, app tab **5. Success measures** |
| §13 Out of scope (OmniChat FAQ, bookings, class-schedule chat) | deliberately not built |

**Design choice.** The LLM does the language work (reading messy notes, summarising, drafting). The Action Standards, due dates, overdue flags and escalations are plain Python rules, so they are predictable, testable and can be audited against the brief's table. Every recommended action records the rule it came from.

Runs on a free Colab T4 GPU with a local open-source model (Ollama `qwen3:8b`), so no parent data leaves the runtime.

## 1. Install Ollama and Python libraries

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y pciutils zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain==1.4.3 langchain-ollama==1.1.0 langchain-chroma==1.1.0 langchain-text-splitters gradio==6.29.0

## 2. Start Ollama and pull the models
`qwen3:8b` (~5 GB) handles structured JSON output well and fits the T4. To try another model, change `MODEL` in §3 and pull it here.

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

!ollama pull qwen3:8b
!ollama pull nomic-embed-text

## 3. Configuration
Centre operating hours, the CST team and owner rules are listed in the brief (§15) as decisions LCentral still has to confirm. They are set here so they are easy to change.

`NOW_OVERRIDE` lets you freeze the clock for a demo (e.g. `"2026-10-17 15:00"`). Leave it as `None` to use the real Singapore time.

In [ ]:
import os, re, json, uuid, shutil, sqlite3, difflib
from contextlib import closing
from datetime import datetime, date, time as dtime, timedelta
from typing import Literal, Optional
from zoneinfo import ZoneInfo
import pandas as pd

try:
    import google.colab  # noqa: F401
    REPO_DIR = "/content/bc3415"
    if not os.path.isdir(REPO_DIR):
        !git clone https://github.com/kingdonkatsu/bc3415.git {REPO_DIR}
    DATA_DIR = REPO_DIR
    WORK_DIR = "/content/work"
except ImportError:
    DATA_DIR = os.path.abspath("data")
    WORK_DIR = os.path.abspath("work")

LEAD_DIR = os.path.join(DATA_DIR, "lead_agent")
CONTENT_DIR = os.path.join(LEAD_DIR, "content_library")
SEED_FILE = os.path.join(LEAD_DIR, "seed_leads.json")
TIMETABLE_FILE = os.path.join(DATA_DIR, "timetable.csv")
DRIVE_DB = os.path.join(LEAD_DIR, "leads.db")
DB_PATH = os.path.join(WORK_DIR, "leads.db")
os.makedirs(WORK_DIR, exist_ok=True)

CENTRE_NAME = "LCentral"
CST_TEAM = ["Aisyah", "Ben", "Clara"]          # Centre Success Team members (owners)
CENTRE_MANAGER = "Centre Manager"              # receives escalations
TZ = ZoneInfo("Asia/Singapore")

# Centre operating hours, Monday=0 ... Sunday=6. None = closed. (Brief §15: to be confirmed by LCentral)
OPERATING_HOURS = {0: None,
                   1: ("14:00", "21:00"), 2: ("14:00", "21:00"), 3: ("14:00", "21:00"), 4: ("14:00", "21:00"),
                   5: ("09:00", "18:00"), 6: ("09:00", "18:00")}

NOW_OVERRIDE = None   # e.g. "2026-10-17 15:00" to freeze the clock for a demo

MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"

def now():
    if NOW_OVERRIDE:
        return datetime.strptime(NOW_OVERRIDE, "%Y-%m-%d %H:%M")
    return datetime.now(TZ).replace(tzinfo=None, second=0, microsecond=0)

def fmt(dt):
    return dt.strftime("%Y-%m-%d %H:%M") if dt else None

def parse_dt(s):
    if not s:
        return None
    if isinstance(s, datetime):
        return s
    for f in ("%Y-%m-%d %H:%M", "%Y-%m-%d"):
        try:
            return datetime.strptime(str(s).strip(), f)
        except ValueError:
            pass
    return None

## 4. Check the data folder
Expected layout (inside `data/`):
```
timetable.csv                     class list, used for class recommendations
lead_agent/seed_leads.json        demo leads in every stage of the journey
lead_agent/content_library/*.md   approved content the drafts may use (brief §8)
```

In [ ]:
REQUIRED = [TIMETABLE_FILE, SEED_FILE, CONTENT_DIR]
missing = [p for p in REQUIRED if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing: {missing}")
TIMETABLE = pd.read_csv(TIMETABLE_FILE)
print("data folder OK:", DATA_DIR)
print("content library:", sorted(f for f in os.listdir(CONTENT_DIR) if f.endswith(".md")))

## 5. LCentral Action Standards engine (brief §6)
The brief says the Agent **must not invent conversion rules**: LCentral defines the standards, the Agent applies them to each lead. So this part is deterministic code, not the LLM.

| Lead situation | Action Standard | Implemented as |
|---|---|---|
| New enquiry | Respond within 4 hours during operating hours; outside hours by the next operating day | `respond_enquiry` |
| No response from parent | 1st follow-up within 1 day; 2nd 2–3 days later; final 5–7 days later | `followup_1/2/final` (due at the end of each window) |
| Assessment interest / booking | Not booked: follow up within 1 day. Booked: confirm immediately; reminder 1 day before | `book_assessment`, `confirm_assessment`, `reminder_assessment` |
| Assessment completed | Follow up same day with recommendation and next steps | `assessment_followup` (priority) |
| Trial booking | Confirm immediately; reminder 1 day before | `confirm_trial`, `reminder_trial` |
| Trial completed | Follow up same day | `trial_followup` (priority) |
| Parent requests later contact | Follow up on the agreed date; if no response, resume the sequence | `contact_after` overrides the due date of every follow-up |
| Considering / undecided | Within 2–3 days; if still considering, again within 5–7 days | `considering_followup` |
| Stale lead | 14 days without response after prescribed attempts, unless a future date was agreed | stale list + `review_nonresponsive` |
| Overdue action | Flag; escalate to Centre Manager after more than 1 operating day; assessments/trials first | board flags + escalation |
| Non-responsive / declined | Stop follow-up / close with reason | outcomes in tab 3 |

Two gaps in the brief's table are filled with clearly labelled assumptions for LCentral to confirm: *trial interest but not booked* (treated like assessment interest, 1 day) and *still considering after two follow-ups* (CST agrees a decision date). Public holidays are not modelled.

In [ ]:
# ---- operating-hours calendar ----
def _hm(s):
    h, m = s.split(":")
    return dtime(int(h), int(m))

def op_window(d):
    hrs = OPERATING_HOURS.get(d.weekday())
    if not hrs:
        return None
    return datetime.combine(d, _hm(hrs[0])), datetime.combine(d, _hm(hrs[1]))

def is_open(dt):
    w = op_window(dt.date())
    return bool(w) and w[0] <= dt < w[1]

def next_operating_day(d):
    for _ in range(14):
        if op_window(d):
            return d
        d += timedelta(days=1)
    raise ValueError("no operating day in the next 2 weeks - check OPERATING_HOURS")

def close_of(d):
    """Closing time of day d, or of the next operating day if the centre is closed on d."""
    return op_window(next_operating_day(d))[1]

def add_operating_hours(dt, hours):
    remaining, cur = timedelta(hours=hours), dt
    while True:
        w = op_window(cur.date())
        if w and cur < w[1]:
            start = max(cur, w[0])
            if start + remaining <= w[1]:
                return start + remaining
            remaining -= w[1] - start
        cur = datetime.combine(cur.date() + timedelta(days=1), dtime(0))

def end_of_next_operating_day(dt):
    """'By the next operating day': the close of the first operating day that opens after dt."""
    w = op_window(dt.date())
    if w and dt < w[0]:
        return w[1]
    return close_of(dt.date() + timedelta(days=1))

def operating_day_before(dt):
    d = dt.date() - timedelta(days=1)
    for _ in range(14):
        if op_window(d):
            return op_window(d)[1]
        d -= timedelta(days=1)

def is_escalated(due, at):
    """Overdue for more than 1 operating day -> escalate to the Centre Manager."""
    return at > close_of(due.date() + timedelta(days=1))

# ---- lead stages ----
NEW, ENGAGING, UNCLEAR = "New enquiry", "Engaging", "Unclear (LCAR import)"
A_INT, A_BOOKED, A_DONE = "Assessment interest", "Assessment booked", "Assessment completed"
T_INT, T_BOOKED, T_DONE = "Trial interest", "Trial booked", "Trial completed"
CONSIDERING, ENROLLED, CLOSED, LOST = "Considering", "Enrolled", "Closed - declined", "Lost - non-responsive"
STAGES = [NEW, ENGAGING, A_INT, A_BOOKED, A_DONE, T_INT, T_BOOKED, T_DONE, CONSIDERING, ENROLLED, CLOSED, LOST, UNCLEAR]
FINAL_STAGES = {ENROLLED, CLOSED, LOST}
HIGH_INTENT = {A_BOOKED, A_DONE, T_INT, T_BOOKED, T_DONE, CONSIDERING}

# What happened in an interaction (the LLM picks one; the code maps it to a stage)
EVENTS = {
    "new_enquiry": "First contact from a parent",
    "info_exchanged": "We shared information / answered questions; ball is in the parent's court",
    "requested_later_contact": "Parent asked to be contacted later, nothing else decided",
    "assessment_interest": "Parent wants an assessment but it is not booked yet",
    "assessment_booked": "Assessment date agreed",
    "assessment_completed": "Child has done the assessment",
    "trial_interest": "Parent wants a trial but it is not booked yet",
    "trial_booked": "Trial date agreed",
    "trial_completed": "Child has done the trial",
    "considering": "Parent is undecided / discussing / thinking about enrolment",
    "no_answer": "Staff tried to contact the parent and got no reply",
    "enrolled": "Parent confirmed enrolment",
    "declined": "Parent clearly said they will not proceed",
    "unclear": "Not enough information to tell",
}
EVENT_STAGE = {"new_enquiry": NEW, "assessment_interest": A_INT, "assessment_booked": A_BOOKED,
               "assessment_completed": A_DONE, "trial_interest": T_INT, "trial_booked": T_BOOKED,
               "trial_completed": T_DONE, "considering": CONSIDERING, "enrolled": ENROLLED, "declined": CLOSED}

SEQUENCE = ["followup_1", "followup_2", "followup_final"]
BOOKING_ACTIONS = {"confirm_assessment", "reminder_assessment", "confirm_trial", "reminder_trial"}
OUTCOME_ACTIONS = {"record_assessment_outcome", "record_trial_outcome"}

def next_stage(current, event):
    if event in EVENT_STAGE:
        return EVENT_STAGE[event]
    if event in ("info_exchanged", "requested_later_contact") and current in (None, "", NEW, UNCLEAR):
        return ENGAGING
    if event == "unclear" and not current:
        return UNCLEAR
    return current

def plan(lead, event, at, prev_action=None):
    """Apply LCentral Action Standards to one lead after an event. Returns a list of actions."""
    acts = []
    def add(kind, due, rule, priority=2):
        acts.append({"action_type": kind, "due_at": due, "rule": rule, "priority": priority})
    stage = lead.get("stage")

    if event == "review":                       # lead has no NBA: re-derive from its stage
        back = {NEW: "new_enquiry", ENGAGING: "info_exchanged", A_INT: "assessment_interest",
                A_BOOKED: "assessment_booked", A_DONE: "assessment_completed", T_INT: "trial_interest",
                T_BOOKED: "trial_booked", T_DONE: "trial_completed", CONSIDERING: "considering"}
        event = back.get(stage, "unclear")
        if missing_fields(lead) and event in ("info_exchanged", "unclear"):
            event = "unclear"

    if event == "new_enquiry":
        if is_open(at):
            add("respond_enquiry", add_operating_hours(at, 4), "New enquiry: respond within 4 hours during operating hours.", 1)
        else:
            add("respond_enquiry", end_of_next_operating_day(at), "New enquiry outside operating hours: respond by the next operating day.", 1)

    elif event in ("info_exchanged", "requested_later_contact"):
        if stage not in (A_BOOKED, T_BOOKED):    # a booking already defines the next step
            add("followup_1", close_of(at.date() + timedelta(days=1)), "No response from parent: 1st follow-up within 1 day.")

    elif event == "followup_sent":               # CST sent a message, no reply yet
        n = SEQUENCE.index(prev_action) + 1 if prev_action in SEQUENCE else 0
        lead["nr_attempts"] = n
        if n == 0:
            add("followup_1", close_of(at.date() + timedelta(days=1)), "No response from parent: 1st follow-up within 1 day.")
        elif n == 1:
            add("followup_2", close_of(at.date() + timedelta(days=3)), "No response: 2nd follow-up 2-3 days after the 1st.")
        elif n == 2:
            add("followup_final", close_of(at.date() + timedelta(days=7)), "No response: final follow-up 5-7 days after the 2nd.")
        else:
            last = parse_dt(lead.get("last_parent_response_at")) or at
            add("review_nonresponsive", max(close_of(last.date() + timedelta(days=14)), close_of(at.date())),
                "Prescribed attempts completed: stop active follow-up. Stale after 14 days without response.", 3)

    elif event == "assessment_interest":
        add("book_assessment", close_of(at.date() + timedelta(days=1)), "Assessment interest, not booked: follow up within 1 day.")

    elif event in ("assessment_booked", "trial_booked"):
        kind = "assessment" if event == "assessment_booked" else "trial"
        add(f"confirm_{kind}", add_operating_hours(at, 1), f"{kind.title()} booked: confirm immediately.", 1)
        when = parse_dt(lead.get(f"{kind}_at"))
        if when:
            rem = operating_day_before(when)
            if rem and rem > at:
                add(f"reminder_{kind}", rem, f"Reminder 1 day before the {kind}.", 2)

    elif event in ("assessment_completed", "trial_completed"):
        kind = "assessment" if event == "assessment_completed" else "trial"
        due = close_of(at.date())
        if due <= at:
            due = end_of_next_operating_day(at)
        rule = ("Assessment completed: follow up same day with a programme/class recommendation and clear next steps."
                if kind == "assessment" else "Trial completed: follow up same day.")
        add(f"{kind}_followup", due, rule, 1)

    elif event == "trial_interest":
        add("book_trial", close_of(at.date() + timedelta(days=1)),
            "ASSUMPTION (not in the Action Standards table, to confirm with LCentral): trial interest treated like assessment interest - follow up within 1 day.")

    elif event == "considering":
        c = lead.get("considering_fu") or 1
        if c <= 1:
            add("considering_followup", close_of(at.date() + timedelta(days=3)), "Parent considering: follow up within 2-3 days.")
        elif c == 2:
            add("considering_followup", close_of(at.date() + timedelta(days=7)), "Still considering: follow up again within 5-7 days.")
        else:
            add("agree_decision_date", close_of(at.date() + timedelta(days=3)),
                "ASSUMPTION (beyond the Action Standards): after two 'considering' follow-ups, CST agrees a decision date with the parent.", 3)

    elif event == "unclear":
        add("clarify", close_of(at.date() + timedelta(days=1)),
            "Not enough information to recommend an action: clarify with the parent rather than guess.")

    # Parent-specific commitments override the default cadence (brief §6)
    ca = parse_dt(lead.get("contact_after"))
    if ca and ca > at:
        for a in acts:
            if a["action_type"] not in BOOKING_ACTIONS:
                a["due_at"] = close_of(ca.date())
                a["rule"] += f" Parent asked to be contacted on/after {ca:%d %b}: their commitment overrides the default cadence."
    return acts

# ---- what is missing from a lead record ----
def missing_fields(lead):
    need = {"level": "child's level", "subjects": "subject(s)", "phone": "parent's contact number"}
    out = [label for f, label in need.items() if not (lead.get(f) or "").strip()]
    if lead.get("stage") in (A_DONE, T_DONE, CONSIDERING) and not (lead.get("preferred_timing") or "").strip():
        out.append("preferred class timing")
    return out

def class_options(lead, n=3):
    lvl = (lead.get("level") or "").strip()
    subs = [s.strip().lower() for s in (lead.get("subjects") or "").split(";") if s.strip()]
    df = TIMETABLE[(TIMETABLE["level"] == lvl) & (TIMETABLE["capacity"] > TIMETABLE["enrolled"])]
    if subs:
        df = df[df["subject"].str.lower().isin(subs)]
    pref = (lead.get("preferred_timing") or "").lower()
    df = df.assign(match=df["day"].str.lower().apply(lambda d: d in pref or d[:3] in pref))
    df = df.sort_values("match", ascending=False).head(n)
    return [f"{r.class_id} {r.subject} {r.level} {r.day} {r.start_time}-{r.end_time} ({r.tutor}, {r.capacity - r.enrolled} seats)"
            for r in df.itertuples()]

def describe(kind, lead):
    """Turn an action type into a concrete instruction for this lead."""
    who = lead.get("parent_name") or "parent"
    child = lead.get("child_name") or "the child"
    prog = " ".join(x for x in [lead.get("level"), (lead.get("subjects") or "").replace(";", "/")] if x) or "programme"
    need = lead.get("learning_needs") or lead.get("parent_goals_concerns")
    concerns = "; ".join(x for x in [lead.get("budget_concern"), lead.get("objections"), lead.get("decision_makers") and f"decision with {lead['decision_makers']}"] if x)
    opts = class_options(lead)
    when_a, when_t = lead.get("assessment_at"), lead.get("trial_at")
    d = {
        "respond_enquiry": f"Reply to {who}'s enquiry about {prog}. Understand {child}'s needs and offer an assessment.",
        "followup_1": f"1st follow-up with {who} (no reply yet). Use a relevant hook, not 'just checking in'.",
        "followup_2": f"2nd follow-up with {who}. Try a different angle linked to {need or 'the child’s needs'}.",
        "followup_final": f"Final follow-up with {who}. Make the next step easy (e.g. a short assessment).",
        "review_nonresponsive": f"All follow-ups done with no reply from {who}. Stop active follow-up; close as non-responsive when stale (keep for future re-engagement).",
        "book_assessment": f"Offer {who} assessment slots for {child} ({prog}) and book one.",
        "confirm_assessment": f"Confirm {child}'s assessment on {when_a} with {who}.",
        "reminder_assessment": f"Send {who} a reminder for the assessment on {when_a}.",
        "record_assessment_outcome": f"Assessment was scheduled for {when_a}. Record the outcome, then follow up same day.",
        "assessment_followup": f"Same-day follow-up with {who}: share assessment findings, recommend a class, propose trial/enrolment.",
        "confirm_trial": f"Confirm {child}'s trial on {when_t} with {who}.",
        "reminder_trial": f"Send {who} a reminder for the trial on {when_t}.",
        "record_trial_outcome": f"Trial was scheduled for {when_t}. Record how it went, then follow up same day.",
        "trial_followup": f"Same-day follow-up with {who}: ask how {child} found the trial, address concerns, propose enrolment.",
        "book_trial": f"Offer {who} a trial slot for {child} ({prog}) and book it.",
        "considering_followup": f"Follow up with {who}, who is still deciding.",
        "agree_decision_date": f"{who} has been considering for a while. Agree a clear decision date.",
        "clarify": f"Record is too thin to act on. Contact {who} to find out: {', '.join(missing_fields(lead)) or 'what they need and the next step'}.",
    }.get(kind, kind)
    if need and kind not in BOOKING_ACTIONS:
        d += f" Need: {need}."
    if concerns and kind in ("considering_followup", "trial_followup", "assessment_followup", "agree_decision_date"):
        d += f" Address: {concerns}."
    if opts and kind in ("assessment_followup", "trial_followup", "considering_followup", "book_trial", "respond_enquiry"):
        d += " Class options: " + " | ".join(opts) + "."
    miss = missing_fields(lead)
    if miss and kind not in {"clarify", "review_nonresponsive"} | BOOKING_ACTIONS:
        d += f" Also obtain: {', '.join(miss)}."
    return d

### 5a. Rule tests
These check the engine against the brief's table without any AI. Run them after changing `OPERATING_HOURS` or the rules.

In [ ]:
def _t(s):
    return datetime.strptime(s, "%Y-%m-%d %H:%M")

def run_rule_tests():
    # 2026-10-13 is a Tuesday (open 14:00-21:00); Monday 2026-10-12 is closed
    L = lambda **k: {"stage": None, **k}
    a = plan(L(), "new_enquiry", _t("2026-10-13 15:00"))[0]
    assert a["due_at"] == _t("2026-10-13 19:00"), a                    # +4 operating hours
    a = plan(L(), "new_enquiry", _t("2026-10-13 19:30"))[0]
    assert a["due_at"] == _t("2026-10-14 16:30"), a                    # 1.5h today + 2.5h tomorrow
    a = plan(L(), "new_enquiry", _t("2026-10-12 10:00"))[0]
    assert a["due_at"] == _t("2026-10-13 21:00"), a                    # Monday closed -> by Tuesday close
    a = plan(L(), "new_enquiry", _t("2026-10-13 08:00"))[0]
    assert a["due_at"] == _t("2026-10-13 21:00"), a                    # before opening -> same operating day
    seq = [plan(L(), "followup_sent", _t("2026-10-14 15:00"), p)[0] for p in [None] + SEQUENCE]
    assert [s["action_type"] for s in seq] == ["followup_1", "followup_2", "followup_final", "review_nonresponsive"]
    assert seq[0]["due_at"].date() == date(2026, 10, 15) and seq[1]["due_at"].date() == date(2026, 10, 17)
    assert seq[2]["due_at"].date() == date(2026, 10, 21)
    acts = plan(L(assessment_at="2026-10-17 10:00"), "assessment_booked", _t("2026-10-14 15:00"))
    assert {x["action_type"] for x in acts} == {"confirm_assessment", "reminder_assessment"}
    assert [x for x in acts if x["action_type"] == "reminder_assessment"][0]["due_at"] == _t("2026-10-16 21:00")
    a = plan(L(), "assessment_completed", _t("2026-10-17 11:00"))[0]
    assert a["due_at"] == _t("2026-10-17 18:00") and a["priority"] == 1      # same day
    a = plan(L(contact_after="2026-10-25"), "assessment_interest", _t("2026-10-14 15:00"))[0]
    assert a["due_at"].date() == date(2026, 10, 25) and "overrides" in a["rule"]   # parent commitment wins
    a = plan(L(considering_fu=2), "considering", _t("2026-10-14 15:00"))[0]
    assert a["due_at"].date() == date(2026, 10, 21)
    assert not is_escalated(_t("2026-10-14 15:00"), _t("2026-10-15 20:00"))
    assert is_escalated(_t("2026-10-14 15:00"), _t("2026-10-15 21:30"))
    assert plan(L(), "unclear", _t("2026-10-14 15:00"))[0]["action_type"] == "clarify"
    print("all Action Standards tests passed")

run_rule_tests()

## 6. Lead database
A simple stand-in for LCAR (brief §11: direct LCAR integration is not needed for the prototype). SQLite runs on a local copy and is copied back to Drive after every change.

* `leads` – one structured record per lead (the fields from brief §1.2)
* `interactions` – every note, outcome and sent message, raw text kept alongside what the AI extracted
* `actions` – Next Best Actions with owner, due date, priority, the rule they came from, status and outcome
* `drafts` – re-engagement drafts, the CST's final version and the edit rate (brief §9 feedback signals)

In [ ]:
LEAD_FIELDS = ["lead_id", "parent_name", "phone", "child_name", "level", "subjects", "learning_needs",
               "parent_goals_concerns", "preferred_timing", "intended_start", "budget_concern", "objections",
               "decision_makers", "previous_tuition", "constraints", "preferred_next_step", "owner", "stage",
               "assessment_at", "trial_at", "contact_after", "nr_attempts", "considering_fu",
               "last_parent_response_at", "last_contact_at", "created_at", "updated_at", "source",
               "closed_reason", "do_not_contact"]
LIST_FIELDS = {"subjects", "learning_needs", "parent_goals_concerns", "objections"}

SCHEMA = f"""
CREATE TABLE IF NOT EXISTS leads ({', '.join(f + (' TEXT PRIMARY KEY' if f == 'lead_id' else ' INTEGER DEFAULT 0' if f in ('nr_attempts', 'considering_fu', 'do_not_contact') else ' TEXT') for f in LEAD_FIELDS)});
CREATE TABLE IF NOT EXISTS interactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT, lead_id TEXT, at TEXT, author TEXT, kind TEXT, event TEXT,
    parent_responded INTEGER, raw_note TEXT, extracted TEXT, summary TEXT);
CREATE TABLE IF NOT EXISTS actions (
    id INTEGER PRIMARY KEY AUTOINCREMENT, lead_id TEXT, action_type TEXT, description TEXT, rule TEXT,
    due_at TEXT, recommended_due_at TEXT, priority INTEGER, owner TEXT, status TEXT, outcome TEXT,
    created_at TEXT, completed_at TEXT, overridden INTEGER DEFAULT 0);
CREATE TABLE IF NOT EXISTS drafts (
    id INTEGER PRIMARY KEY AUTOINCREMENT, lead_id TEXT, action_id INTEGER, created_at TEXT, purpose TEXT,
    draft TEXT, final TEXT, sources TEXT, warnings TEXT, edit_ratio REAL, status TEXT, sent_at TEXT);
"""

def persist_db():
    if os.path.abspath(DB_PATH) != os.path.abspath(DRIVE_DB):
        shutil.copy(DB_PATH, DRIVE_DB)

def q(sql, params=()):
    with closing(sqlite3.connect(DB_PATH)) as con:
        con.row_factory = sqlite3.Row
        return [dict(r) for r in con.execute(sql, params).fetchall()]

def ex(sql, params=()):
    with closing(sqlite3.connect(DB_PATH)) as con:
        cur = con.execute(sql, params)
        con.commit()
        return cur.lastrowid

def get_lead(lead_id):
    rows = q("SELECT * FROM leads WHERE lead_id = ?", (lead_id,))
    return rows[0] if rows else None

def save_lead(lead):
    lead["updated_at"] = fmt(now())
    cols = [f for f in LEAD_FIELDS if f in lead]
    ex(f"INSERT OR REPLACE INTO leads ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})",
       [lead[c] for c in cols])

def new_lead_id():
    n = q("SELECT COUNT(*) n FROM leads")[0]["n"]
    while get_lead(f"L{n + 1:03d}"):
        n += 1
    return f"L{n + 1:03d}"

def assign_owner():
    """Owner rule (brief §15, to be confirmed): the CST member with the fewest open actions."""
    load = {m: 0 for m in CST_TEAM}
    for r in q("SELECT owner, COUNT(*) n FROM actions WHERE status = 'open' GROUP BY owner"):
        if r["owner"] in load:
            load[r["owner"]] = r["n"]
    return min(load, key=load.get)

def open_actions(lead_id=None):
    if lead_id:
        return q("SELECT * FROM actions WHERE status = 'open' AND lead_id = ? ORDER BY due_at", (lead_id,))
    return q("SELECT * FROM actions WHERE status = 'open' ORDER BY due_at")

def add_actions(lead, acts, at):
    ids = []
    for a in acts:
        ids.append(ex("""INSERT INTO actions (lead_id, action_type, description, rule, due_at, recommended_due_at,
                         priority, owner, status, created_at) VALUES (?,?,?,?,?,?,?,?, 'open', ?)""",
                      (lead["lead_id"], a["action_type"], describe(a["action_type"], lead), a["rule"],
                       fmt(a["due_at"]), fmt(a["due_at"]), a["priority"], lead["owner"], fmt(at))))
    return ids

def close_open_actions(lead_id, outcome, at, keep_booking=False):
    for a in open_actions(lead_id):
        if keep_booking and a["action_type"] in BOOKING_ACTIONS:
            continue
        ex("UPDATE actions SET status = 'superseded', outcome = ?, completed_at = ? WHERE id = ?", (outcome, fmt(at), a["id"]))

def log_interaction(lead_id, at, author, kind, event, parent_responded, raw_note, extracted=None, summary=None):
    return ex("""INSERT INTO interactions (lead_id, at, author, kind, event, parent_responded, raw_note, extracted, summary)
                 VALUES (?,?,?,?,?,?,?,?,?)""",
              (lead_id, fmt(at), author, kind, event, int(bool(parent_responded)), raw_note,
               json.dumps(extracted) if extracted else None, summary))

def init_db():
    if os.path.exists(DRIVE_DB) and not os.path.exists(DB_PATH):
        shutil.copy(DRIVE_DB, DB_PATH)
    with closing(sqlite3.connect(DB_PATH)) as con:
        con.executescript(SCHEMA)
        con.commit()

## 7. Models and the approved content library (brief §8)
Drafts may only use programme facts, curriculum insights, objection responses, offers and testimonials from `content_library/`. Each `##` section is one approved snippet, embedded into Chroma and cited by id (`file#section`). The sample content is labelled as such and must be replaced or approved by LCentral before any parent-facing use.

In [ ]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_chroma import Chroma

llm = ChatOllama(model=MODEL, temperature=0, reasoning=False, num_ctx=8192)
llm_draft = ChatOllama(model=MODEL, temperature=0.4, reasoning=False, num_ctx=8192)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

def load_content_library():
    snippets = []
    for name in sorted(os.listdir(CONTENT_DIR)):
        if not name.endswith(".md") or name.lower() == "readme.md":
            continue
        text = open(os.path.join(CONTENT_DIR, name), encoding="utf-8").read()
        for block in re.split(r"\n(?=## )", text):
            if not block.startswith("## "):
                continue
            title, _, body = block[3:].partition("\n")
            if body.strip():
                sid = f"{name[:-3]}#{title.strip()}"
                snippets.append(Document(page_content=f"{title.strip()}\n{body.strip()}", metadata={"id": sid, "file": name}))
    return snippets

SNIPPETS = load_content_library()
vectorstore = Chroma(collection_name=f"content_{uuid.uuid4().hex[:8]}", embedding_function=embeddings)
vectorstore.add_documents(SNIPPETS)
print(len(SNIPPETS), "approved snippets indexed")

## 8. AI capture: from a rough note to a structured lead (brief §5A)
The CST types or pastes a short note after any parent interaction. The LLM extracts the fields from brief §1.2 and says what happened (the *event*). The code then:
* normalises the level and checks dates,
* compares against the existing record and flags **conflicts** (e.g. note says P5, record says P4),
* lists **missing** information so the CST can ask for it next time.

Nothing is saved until the CST has reviewed and confirmed it.

In [ ]:
class LeadNote(BaseModel):
    parent_name: Optional[str] = Field(None, description="Parent's name or title as written, e.g. 'Mrs Tan'")
    child_name: Optional[str] = Field(None, description="Child's name if mentioned")
    phone: Optional[str] = Field(None, description="Parent's phone number if mentioned")
    level: Optional[str] = Field(None, description="School level, e.g. P4 or Sec 3")
    subjects: list[str] = Field(default_factory=list, description="Subjects, e.g. ['English']")
    learning_needs: list[str] = Field(default_factory=list, description="Specific weak areas or learning needs, e.g. ['composition']")
    parent_goals_concerns: list[str] = Field(default_factory=list, description="What the parent wants or worries about")
    preferred_timing: Optional[str] = Field(None, description="Preferred class days/times, e.g. 'Saturday afternoon'")
    intended_start: Optional[str] = Field(None, description="When they want to start")
    budget_concern: Optional[str] = Field(None, description="Any pricing or budget concern")
    objections: list[str] = Field(default_factory=list, description="Objections or uncertainties")
    decision_makers: Optional[str] = Field(None, description="Other people who must agree, e.g. 'husband'")
    previous_tuition: Optional[str] = Field(None, description="Previous or current tuition experience")
    constraints: Optional[str] = Field(None, description="Constraints such as travel or exams")
    preferred_next_step: Optional[str] = Field(None, description="Next step the parent prefers, e.g. 'assessment'")
    event: Literal[tuple(EVENTS)] = Field(description="What happened in THIS interaction")
    parent_responded: bool = Field(True, description="False only if staff tried to reach the parent and got no reply")
    assessment_datetime: Optional[str] = Field(None, description="Agreed assessment date and time, YYYY-MM-DD HH:MM")
    trial_datetime: Optional[str] = Field(None, description="Agreed trial date and time, YYYY-MM-DD HH:MM")
    contact_after_date: Optional[str] = Field(None, description="Earliest date the parent agreed to be contacted, YYYY-MM-DD")
    declined_reason: Optional[str] = Field(None, description="Why the parent declined, if they did")
    summary: str = Field(description="One plain sentence summarising the interaction")

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def calendar_text(start, n=21):
    return "; ".join(f"{DAYS[d.weekday()][:3]} {d:%d %b} = {d.isoformat()}" for d in (start + timedelta(days=i) for i in range(n)))

CAPTURE_PROMPT = """You help the Centre Success Team (CST) of {centre}, a Singapore tuition centre, turn a staff member's rough note about a parent interaction into a structured lead record.
Today is {today}. Calendar for the coming weeks: {calendar}

Rules:
- Extract only what the NEW NOTE says. Use null or [] when something is not stated. Never guess or fill gaps.
- level: Singapore school level, P1-P6 or Sec 1-Sec 5 ("Pri 4" -> "P4", "S3" -> "Sec 3").
- event: what happened in THIS interaction. Choose exactly one:
{events}
- If the note mentions a milestone (e.g. wants an assessment) AND a request to be contacted later, choose the milestone as event and put the date in contact_after_date.
- contact_after_date: earliest date the parent agreed to be contacted, YYYY-MM-DD, using the calendar above. "Travelling until 18 Nov" or "back on 18 Nov, message after" means the day after: 19 Nov. "Call me next Tuesday" means that Tuesday.
- assessment_datetime / trial_datetime: only if a specific slot was agreed, YYYY-MM-DD HH:MM.
- summary: one plain sentence a colleague can read in five seconds."""

def norm_level(level):
    s = (level or "").strip().lower().replace(".", "")
    m = re.match(r"^(p|pri|primary)\s*([1-6])$", s)
    if m:
        return f"P{m.group(2)}"
    m = re.match(r"^(s|sec|secondary)\s*([1-5])$", s)
    if m:
        return f"Sec {m.group(2)}"
    return (level or "").strip() or None

def call_structured(schema, system, user, model=None):
    model = (model or llm).with_structured_output(schema, method="json_schema")
    last = None
    for _ in range(2):
        try:
            return model.invoke([SystemMessage(system), HumanMessage(user)])
        except Exception as e:      # malformed JSON from a small model: retry once
            last = e
    raise last

REVIEW_FIELDS = ["parent_name", "child_name", "phone", "level", "subjects", "learning_needs", "parent_goals_concerns",
                 "preferred_timing", "intended_start", "budget_concern", "objections", "decision_makers",
                 "previous_tuition", "constraints", "preferred_next_step", "assessment_at", "trial_at",
                 "contact_after", "declined_reason"]

def understand_note(note, lead_id=None):
    t = now()
    lead = get_lead(lead_id) if lead_id else {}
    known = {k: v for k, v in (lead or {}).items() if k in REVIEW_FIELDS + ["stage"] and v}
    system = CAPTURE_PROMPT.format(centre=CENTRE_NAME, today=f"{DAYS[t.weekday()]} {t:%d %B %Y %H:%M}",
                                   calendar=calendar_text(t.date()),
                                   events="\n".join(f"  {k}: {v}" for k, v in EVENTS.items()))
    user = (f"EXISTING RECORD (context only, may be outdated): {json.dumps(known)}\n\n" if known else "") + f"NEW NOTE: {note}"
    warnings = []
    try:
        x = call_structured(LeadNote, system, user).model_dump()
    except Exception as e:
        x = LeadNote(event="unclear", summary=note[:150]).model_dump()
        warnings.append(f"AI could not read this note ({type(e).__name__}); please fill the fields in by hand.")

    fields = {k: x.get(k) for k in REVIEW_FIELDS if k in x}
    fields["level"] = norm_level(x.get("level"))
    fields["assessment_at"], fields["trial_at"] = x.get("assessment_datetime"), x.get("trial_datetime")
    fields["contact_after"], fields["declined_reason"] = x.get("contact_after_date"), x.get("declined_reason")
    for f in ("assessment_at", "trial_at", "contact_after"):
        if fields[f]:
            d = parse_dt(fields[f])
            if not d:
                warnings.append(f"{f}: '{fields[f]}' is not a valid date, please correct it.")
            elif d.date() < t.date():
                warnings.append(f"{f}: {fields[f]} is in the past, please check.")
    if fields["contact_after"] and parse_dt(fields["contact_after"]):
        fields["contact_after"] = parse_dt(fields["contact_after"]).strftime("%Y-%m-%d")

    conflicts = []
    for f in ("level", "child_name", "phone", "parent_name"):
        old, new = (lead or {}).get(f), fields.get(f)
        if old and new and re.sub(r"\W", "", str(old).lower()) != re.sub(r"\W", "", str(new).lower()):
            conflicts.append(f"{f}: record says '{old}', note says '{new}'")
    merged = merge_fields(dict(lead or {}), {k: v for k, v in fields.items() if k in LEAD_FIELDS})
    merged["stage"] = next_stage((lead or {}).get("stage"), x["event"])
    return {"lead_id": lead_id, "note": note, "fields": fields, "event": x["event"],
            "parent_responded": x["parent_responded"], "summary": x["summary"], "conflicts": conflicts,
            "missing": missing_fields(merged), "warnings": warnings, "raw": x, "existing": known}

def review_rows(u):
    rows = []
    for f in REVIEW_FIELDS:
        v = u["fields"].get(f)
        v = "; ".join(v) if isinstance(v, list) else (v or "")
        check = ""
        if any(c.startswith(f + ":") for c in u["conflicts"]):
            check = "CONFLICT with record"
        elif not v and u["existing"].get(f):
            check = f"on record: {u['existing'][f]}"
        elif not v and f in ("level", "subjects", "phone"):
            check = "MISSING - please ask"
        rows.append([f, v, check])
    return rows

## 9. The operating loop (brief §5C)
**Interaction → Understand → Capture → Next Best Action → Act → Outcome → Update → Next Best Action**, until the lead is Enrolled, Closed or Lost.

* `apply_capture` – the CST has confirmed what the AI extracted from a note. Update the lead, log the interaction, supersede stale actions and apply the Action Standards.
* `record_outcome` – the CST completed an action. Record the outcome and work out what happens next.
* `refresh_time_based` – things that become due just because time passes (an assessment date has gone by with no outcome recorded).

In [ ]:
def merge_fields(lead, fields):
    for f, v in fields.items():
        if f not in LEAD_FIELDS or v in (None, "", []):
            continue
        if f in LIST_FIELDS:
            old = [x.strip() for x in (lead.get(f) or "").split(";") if x.strip()]
            new = v if isinstance(v, list) else [x.strip() for x in str(v).split(";")]
            for x in new:
                if x and x.lower() not in [o.lower() for o in old]:
                    old.append(x)
            lead[f] = "; ".join(old)
        else:
            lead[f] = str(v).strip()
    return lead

def apply_capture(lead_id, fields, event, note, author, parent_responded=True, extracted=None, at=None, summary=None):
    at = at or now()
    if lead_id:
        lead = get_lead(lead_id)
    else:
        lead = {f: None for f in LEAD_FIELDS}
        lead.update(lead_id=new_lead_id(), created_at=fmt(at), source="Agent capture", nr_attempts=0,
                    considering_fu=0, do_not_contact=0)
        lead["owner"] = author if author in CST_TEAM else assign_owner()
    lead = merge_fields(lead, fields)

    prev_followup = next((a["action_type"] for a in open_actions(lead["lead_id"]) if a["action_type"] in SEQUENCE), None)
    if parent_responded and event != "no_answer":
        lead["last_parent_response_at"] = fmt(at)
        lead["nr_attempts"] = 0
    lead["considering_fu"] = (lead.get("considering_fu") or 0) + 1 if event == "considering" else 0
    lead["stage"] = next_stage(lead.get("stage"), event)
    if event == "declined":
        lead["closed_reason"] = fields.get("declined_reason") or summary or note[:120]
    if event in ("assessment_completed", "trial_completed"):
        lead["contact_after"] = None           # a completed milestone needs same-day follow-up

    iid = log_interaction(lead["lead_id"], at, author, "note", event, parent_responded, note, extracted, summary)
    keep = lead["stage"] in (A_BOOKED, T_BOOKED) and event not in ("assessment_booked", "trial_booked")
    close_open_actions(lead["lead_id"], f"Superseded by interaction #{iid}", at, keep_booking=keep)

    if event == "no_answer":
        acts = plan(lead, "followup_sent", at, prev_followup)
    elif lead["stage"] in FINAL_STAGES:
        acts = []
    else:
        acts = plan(lead, event, at)
    lead["last_contact_at"] = fmt(at)
    save_lead(lead)
    ids = add_actions(lead, acts, at)
    persist_db()
    return get_lead(lead["lead_id"]), ids

OUTCOMES = ["Sent - awaiting reply", "Done - nothing further", "Enrolled", "Declined", "Close as non-responsive"]
FOLLOWUP_LIKE = set(SEQUENCE) | {"respond_enquiry", "book_assessment", "book_trial", "assessment_followup",
                                 "trial_followup", "considering_followup", "agree_decision_date", "clarify"}

def record_outcome(action_id, outcome, note="", author="", at=None):
    at = at or now()
    a = q("SELECT * FROM actions WHERE id = ?", (action_id,))[0]
    lead = get_lead(a["lead_id"])
    ex("UPDATE actions SET status = 'done', outcome = ?, completed_at = ? WHERE id = ?",
       (outcome + (f": {note}" if note else ""), fmt(at), action_id))
    log_interaction(lead["lead_id"], at, author, "outcome", outcome, 0, f"[{a['action_type']}] {outcome}. {note}".strip())
    new_ids = []
    if outcome == "Sent - awaiting reply" and a["action_type"] in FOLLOWUP_LIKE:
        lead["last_contact_at"] = fmt(at)
        close_open_actions(lead["lead_id"], "Superseded: new follow-up scheduled", at, keep_booking=True)
        new_ids = add_actions(lead, plan(lead, "followup_sent", at, a["action_type"]), at)
    elif outcome in ("Enrolled", "Declined", "Close as non-responsive"):
        lead["stage"] = {"Enrolled": ENROLLED, "Declined": CLOSED, "Close as non-responsive": LOST}[outcome]
        if outcome == "Declined":
            lead["closed_reason"] = note or "Not recorded"
        close_open_actions(lead["lead_id"], f"Lead {lead['stage']}", at)
    save_lead(lead)
    persist_db()
    return new_ids

def recommend_for_lead(lead_id, author="system"):
    """For an active lead with no Next Best Action (e.g. an LCAR import)."""
    lead = get_lead(lead_id)
    if lead["stage"] in FINAL_STAGES or open_actions(lead_id):
        return []
    at = now()
    ids = add_actions(lead, plan(lead, "review", at), at)
    persist_db()
    return ids

def override_action(action_id, due_at=None, description=None):
    """CST disagrees with the recommendation. Recorded as a feedback signal (brief §9)."""
    a = q("SELECT * FROM actions WHERE id = ?", (action_id,))[0]
    due = parse_dt(due_at) if due_at else None
    ex("UPDATE actions SET due_at = ?, description = ?, overridden = 1 WHERE id = ?",
       (fmt(due) if due else a["due_at"], description or a["description"], action_id))
    persist_db()

def refresh_time_based(at=None):
    at = at or now()
    for kind, stage in (("assessment", A_BOOKED), ("trial", T_BOOKED)):
        for lead in q(f"SELECT * FROM leads WHERE stage = ? AND {kind}_at IS NOT NULL", (stage,)):
            when = parse_dt(lead[f"{kind}_at"])
            if not when or when + timedelta(hours=2) > at:
                continue
            if any(x["action_type"] == f"record_{kind}_outcome" for x in open_actions(lead["lead_id"])):
                continue
            close_open_actions(lead["lead_id"], f"{kind.title()} date has passed", at)
            add_actions(lead, [{"action_type": f"record_{kind}_outcome", "due_at": close_of(when.date()), "priority": 1,
                                "rule": f"{kind.title()} completed: follow up same day. No outcome has been recorded yet."}], at)
    persist_db()

### 9a. Seed the demo leads
Loads `seed_leads.json`: 12 leads at every stage of the journey plus 3 imported in today's vague LCAR style ("Interested. Follow up."). Times are relative to now, so the board always shows due, overdue, escalated and stale items. Actions are created by the same Action Standards engine, as if each event had happened at that time.

Set `RESET_DEMO_DATA = True` to wipe the database and reseed (e.g. before recording the demo).

In [ ]:
RESET_DEMO_DATA = False

def seed_demo():
    base = now()
    H = lambda h: base + timedelta(hours=h)
    seed = json.load(open(SEED_FILE, encoding="utf-8"))["leads"]
    for s in seed:
        lead = {f: None for f in LEAD_FIELDS}
        lead.update({k: v for k, v in s.items() if k in LEAD_FIELDS})
        lead.update(nr_attempts=s.get("nr_attempts", 0), considering_fu=0, do_not_contact=0,
                    created_at=fmt(H(s.get("created_h", -24))))
        for k in ("assessment_at", "trial_at"):
            if f"{k}_h" in s:                     # put the slot inside operating hours, 1h after opening
                d, step = H(s[f"{k}_h"]).date(), (1 if s[f"{k}_h"] > 0 else -1)
                while not op_window(d):
                    d += timedelta(days=step)
                lead[k] = fmt(op_window(d)[0] + timedelta(hours=1))
        if "last_parent_response_h" in s:
            lead["last_parent_response_at"] = fmt(H(s["last_parent_response_h"]))
        if "contact_after_h" in s:
            lead["contact_after"] = H(s["contact_after_h"]).strftime("%Y-%m-%d")
        last_event, last_at = None, None
        for h in s["history"]:
            at = H(h["h"])
            log_interaction(lead["lead_id"], at, lead["owner"], "note", h["event"], h.get("parent_responded", True), h["note"])
            if h.get("parent_responded", True) and not s.get("last_parent_response_h"):
                lead["last_parent_response_at"] = fmt(at)
            if not s.get("stage"):
                lead["stage"] = next_stage(lead.get("stage"), h["event"])
            if h["event"] == "considering":
                lead["considering_fu"] += 1
            last_event, last_at = h["event"], at
        if s.get("stage"):
            lead["stage"] = s["stage"]
        lead["last_contact_at"] = fmt(last_at)
        save_lead(lead)
        if lead["source"] == "LCAR import" or lead["stage"] in FINAL_STAGES:
            continue                                   # LCAR imports have no NBA yet
        if "seed_event" in s:
            se = s["seed_event"]
            at = H(se["h"])
            log_interaction(lead["lead_id"], at, lead["owner"], "outcome", "Sent - awaiting reply", 0, f"[{se['prev_action']}] Sent - awaiting reply")
            acts = plan(lead, se["event"], at, se["prev_action"])
        else:
            acts = plan(lead, last_event, last_at)
        save_lead(lead)
        add_actions(lead, acts, last_at)
    refresh_time_based()
    persist_db()

def reset_db():
    for p in {DB_PATH, DRIVE_DB}:
        if os.path.exists(p):
            os.remove(p)
    init_db()
    seed_demo()

init_db()
if RESET_DEMO_DATA or not q("SELECT 1 FROM leads LIMIT 1"):
    reset_db()
    print("database seeded")
print(pd.DataFrame(q("SELECT lead_id, parent_name, level, subjects, stage, owner, source FROM leads")).to_string(index=False))

## 10. Action board: making sure actions happen (brief §5C)
What the CST and Centre Manager see each day. Everything here is computed from the database, no AI involved.

In [ ]:
def _days(dt, ref):
    return (ref.date() - dt.date()).days

def board(owner="Everyone"):
    refresh_time_based()
    t = now()
    acts = q("""SELECT a.*, l.parent_name, l.child_name, l.level, l.subjects, l.stage FROM actions a
                JOIN leads l ON l.lead_id = a.lead_id WHERE a.status = 'open' ORDER BY a.priority, a.due_at""")
    leads = q("SELECT * FROM leads")
    if owner in CST_TEAM:
        acts = [a for a in acts if a["owner"] == owner]
        leads = [l for l in leads if l["owner"] == owner]

    def row(a, extra=None):
        due = parse_dt(a["due_at"])
        r = {"Lead": a["lead_id"], "Due": f"{DAYS[due.weekday()][:3]} {due:%d %b %H:%M}", "Priority": {1: "High", 2: "Normal", 3: "Low"}[a["priority"]]}
        r.update(extra or {})
        r.update({"Owner": a["owner"], "Parent": a["parent_name"], "Child": f"{a['child_name'] or ''} {a['level'] or ''} {a['subjects'] or ''}".strip(),
                  "Stage": a["stage"], "Next Best Action": a["description"]})
        return r

    overdue = [row(a, {"Overdue by": f"{_days(parse_dt(a['due_at']), t)} day(s)" if _days(parse_dt(a['due_at']), t) else "today",
                       "Escalate": "Centre Manager" if is_escalated(parse_dt(a["due_at"]), t) else ""})
               for a in acts if parse_dt(a["due_at"]) < t]
    overdue.sort(key=lambda r: (r["Escalate"] == "", r["Priority"] != "High"))
    due_today = [row(a) for a in acts if parse_dt(a["due_at"]) >= t and parse_dt(a["due_at"]).date() == t.date()]
    soon = t + timedelta(hours=24)
    high = [row(a) for a in acts if a["stage"] in HIGH_INTENT and parse_dt(a["due_at"]) <= soon]
    upcoming = [row(a) for a in acts if t.date() < parse_dt(a["due_at"]).date() <= (t + timedelta(days=7)).date()]
    with_action = {a["lead_id"] for a in q("SELECT DISTINCT lead_id FROM actions WHERE status = 'open'")}
    active = [l for l in leads if l["stage"] not in FINAL_STAGES]
    lrow = lambda l, **k: {"Lead": l["lead_id"], "Parent": l["parent_name"], "Child": f"{l['child_name'] or ''} {l['level'] or ''} {l['subjects'] or ''}".strip(),
                           "Stage": l["stage"], "Owner": l["owner"], "Source": l["source"], **k}
    no_nba = [lrow(l, **{"Missing": ", ".join(missing_fields(l))}) for l in active if l["lead_id"] not in with_action]
    a_open = [lrow(l, **{"Assessment was": l["assessment_at"]}) for l in active if l["stage"] == A_BOOKED and parse_dt(l["assessment_at"]) and parse_dt(l["assessment_at"]) < t]
    t_open = [lrow(l, **{"Trial was": l["trial_at"]}) for l in active if l["stage"] == T_BOOKED and parse_dt(l["trial_at"]) and parse_dt(l["trial_at"]) < t]
    stale = []
    for l in active:
        last = parse_dt(l["last_parent_response_at"])
        future = parse_dt(l["contact_after"]) and parse_dt(l["contact_after"]) > t
        if (l["nr_attempts"] or 0) >= 3 and last and (t - last).days >= 14 and not future:
            stale.append(lrow(l, **{"Last reply": l["last_parent_response_at"], "Days silent": (t - last).days}))
    return {"overdue": overdue, "due_today": due_today, "high_intent": high, "assessments_no_outcome": a_open,
            "trials_no_outcome": t_open, "no_nba": no_nba, "stale": stale, "upcoming": upcoming}

def board_summary(b):
    esc = sum(1 for r in b["overdue"] if r["Escalate"])
    return (f"**{len(b['overdue'])}** overdue ({esc} escalated to the Centre Manager) · **{len(b['due_today'])}** due later today · "
            f"**{len(b['high_intent'])}** high-intent needing attention · **{len(b['assessments_no_outcome'])}** assessments and "
            f"**{len(b['trials_no_outcome'])}** trials without outcomes · **{len(b['no_nba'])}** active leads without a Next Best Action · "
            f"**{len(b['stale'])}** stale")

## 11. Parent re-engagement drafts (brief §7–9)
*Action Standards decide **when** to follow up; parent context decides **what** to say.*

The draft uses the lead record, recent interactions, suitable classes with free seats, and approved snippets retrieved from the content library. Code-level guardrails then check it before the CST sees it:
* no generic "just checking in" phrasing,
* every price or percentage must appear in the approved content or lead record,
* no quotations (testimonials may only come from the library, which is empty until LCentral adds real ones),
* length, and the parent's opt-out flag.

The CST edits and sends it themselves. The edit rate is stored as a feedback signal.

In [ ]:
class Draft(BaseModel):
    message: str = Field(description="The WhatsApp message to the parent")
    sources_used: list[str] = Field(default_factory=list, description="ids of APPROVED CONTENT snippets used")

DRAFT_PROMPT = """You draft WhatsApp messages for the Centre Success Team (CST) of {centre}, a Singapore tuition centre. A CST member reviews and edits every draft before sending; you never send anything.
Purpose of this message: {purpose}

Rules:
- Use ONLY facts in LEAD RECORD, RECENT INTERACTIONS, CLASS OPTIONS and APPROVED CONTENT. Never invent prices, dates, results, exam facts, offers or testimonials.
- Never write "just checking in", "just following up" or "are you still interested".
- Make it about this child: refer to their specific need or the parent's concern, and give one genuinely useful thing from APPROVED CONTENT (an insight, a suitable class, an answer to their concern).
- Warm and conversational, like a helpful person at the centre. At most 80 words. No bullet points. At most one emoji.
- End with one easy question that moves to the next step.
- Sign off as {cst} from {centre}.
- sources_used: the [ids] of the APPROVED CONTENT snippets you actually used."""

BANNED_PHRASES = ["just checking in", "just check in", "just following up", "just to follow up", "still interested"]

def draft_context(lead):
    query = " ".join(str(lead.get(f) or "") for f in ("level", "subjects", "learning_needs", "parent_goals_concerns",
                                                     "budget_concern", "objections", "decision_makers", "stage"))
    docs = vectorstore.similarity_search(query, k=5)
    hist = q("SELECT at, kind, raw_note, summary FROM interactions WHERE lead_id = ? ORDER BY at DESC LIMIT 4", (lead["lead_id"],))
    record = {k: v for k, v in lead.items() if v and k in REVIEW_FIELDS + ["stage"]}
    approved = "\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in docs)
    classes = "\n".join(class_options(lead)) or "none listed"
    text = (f"LEAD RECORD: {json.dumps(record)}\n\nRECENT INTERACTIONS (newest first):\n" +
            "\n".join(f"- {h['at']}: {h['summary'] or h['raw_note']}" for h in hist) +
            f"\n\nCLASS OPTIONS (only these have seats):\n{classes}\n\nAPPROVED CONTENT:\n{approved}")
    return text, [d.metadata["id"] for d in docs]

def check_draft(msg, context, lead):
    warns, low = [], msg.lower()
    for p in BANNED_PHRASES:
        if p in low:
            warns.append(f"Generic phrase: '{p}'")
    flat = re.sub(r"\s", "", context.lower())
    for tok in re.findall(r"\$\s?\d[\d,]*(?:\.\d+)?|\d+(?:\.\d+)?\s?%", msg):
        if re.sub(r"\s", "", tok.lower()) not in flat:
            warns.append(f"Unverified figure '{tok}' - not in approved content or the lead record")
    if re.search(r"[\"“”]", msg):
        warns.append("Contains a quotation - testimonials must come from the approved library")
    if len(msg.split()) > 110:
        warns.append(f"Long message ({len(msg.split())} words)")
    if lead.get("do_not_contact"):
        warns.append("Parent has opted out of messages - do not send")
    return warns

def draft_message(lead_id, action_id=None, cst=None):
    lead = get_lead(lead_id)
    if lead.get("do_not_contact"):
        return {"blocked": "Parent has opted out of marketing/re-engagement messages."}
    act = q("SELECT * FROM actions WHERE id = ?", (action_id,))[0] if action_id else (open_actions(lead_id) or [None])[0]
    purpose = act["description"] if act else f"Re-engage a parent who has gone quiet (stage: {lead['stage']})."
    cst = cst or lead["owner"]
    context, retrieved = draft_context(lead)
    system = DRAFT_PROMPT.format(centre=CENTRE_NAME, purpose=purpose, cst=cst)
    try:
        d = call_structured(Draft, system, context, model=llm_draft)
        msg, used = d.message.strip(), [s for s in d.sources_used if s in retrieved]
    except Exception as e:
        return {"blocked": f"Drafting failed ({type(e).__name__}). Try again."}
    warns = check_draft(msg, context, lead)
    did = ex("""INSERT INTO drafts (lead_id, action_id, created_at, purpose, draft, sources, warnings, status)
                VALUES (?,?,?,?,?,?,?, 'drafted')""",
             (lead_id, act["id"] if act else None, fmt(now()), purpose, msg, json.dumps(used), json.dumps(warns)))
    persist_db()
    return {"draft_id": did, "message": msg, "sources": used, "warnings": warns, "purpose": purpose,
            "action_id": act["id"] if act else None}

def mark_sent(draft_id, final_text, author):
    d = q("SELECT * FROM drafts WHERE id = ?", (draft_id,))[0]
    ratio = round(1 - difflib.SequenceMatcher(None, d["draft"], final_text).ratio(), 3)
    ex("UPDATE drafts SET final = ?, edit_ratio = ?, status = 'sent', sent_at = ? WHERE id = ?",
       (final_text, ratio, fmt(now()), draft_id))
    log_interaction(d["lead_id"], now(), author, "message", "message_sent", 0, final_text)
    new_ids = record_outcome(d["action_id"], "Sent - awaiting reply", "re-engagement message", author) if d["action_id"] else []
    persist_db()
    return ratio, new_ids

def summarise_lead(lead_id):
    lead = get_lead(lead_id)
    hist = q("SELECT at, kind, raw_note FROM interactions WHERE lead_id = ? ORDER BY at", (lead_id,))
    text = "\n".join(f"{h['at']} ({h['kind']}): {h['raw_note']}" for h in hist)
    record = {k: v for k, v in lead.items() if v and k in REVIEW_FIELDS + ["stage"]}
    msg = [SystemMessage("Summarise this tuition-centre lead for a busy CST member in at most 60 words: who the child is, "
                         "what the parent needs, where things stand and any concern. Then one line starting 'Watch out:' "
                         "naming missing or conflicting information, or 'Watch out: nothing' if none. Use only the text given."),
           HumanMessage(f"RECORD: {json.dumps(record)}\nHISTORY:\n{text}")]
    try:
        return llm.invoke(msg).content.strip()
    except Exception as e:
        return f"(summary unavailable: {type(e).__name__})"

## 12. Success measures (brief §14)
* **Lead data quality** – how many of the brief §1.2 fields are filled: LCAR-style records vs records captured through the Agent.
* **Follow-up discipline** – share of active leads with Owner + Stage + Next Best Action + Due date; overdue actions; assessments/trials without outcomes.
* **CST feedback** (brief §9) – draft edit rate, discarded drafts, Next Best Action overrides.
* **Re-engagement** – reply rate after a sent draft.

In [ ]:
CAPTURE_FIELDS = ["level", "subjects", "learning_needs", "parent_goals_concerns", "preferred_timing", "intended_start",
                  "budget_concern", "objections", "decision_makers", "previous_tuition", "preferred_next_step"]

def metrics():
    leads = pd.DataFrame(q("SELECT * FROM leads"))
    leads["completeness"] = leads[CAPTURE_FIELDS].apply(lambda r: sum(pd.notna(v) and bool(str(v).strip()) for v in r) / len(CAPTURE_FIELDS), axis=1)
    leads["group"] = leads["source"].apply(lambda s: "LCAR-style record" if s == "LCAR import" else "Captured via Agent")
    quality = leads.groupby("group")["completeness"].agg(["count", "mean"]).rename(columns={"count": "Leads", "mean": "Avg fields filled"})
    quality["Avg fields filled"] = (quality["Avg fields filled"] * 100).round(0).astype(int).astype(str) + "%"

    t = now()
    active = leads[~leads["stage"].isin(FINAL_STAGES)]
    acts = pd.DataFrame(q("SELECT * FROM actions"))
    open_a = acts[acts["status"] == "open"] if len(acts) else acts
    with_nba = active["lead_id"].isin(open_a["lead_id"]) if len(open_a) else pd.Series(False, index=active.index)
    complete = with_nba & active["owner"].notna() & active["stage"].notna()
    overdue = open_a[open_a["due_at"].apply(lambda d: parse_dt(d) < t)] if len(open_a) else open_a
    b = board()
    drafts = pd.DataFrame(q("SELECT * FROM drafts"))
    sent = drafts[drafts["status"] == "sent"] if len(drafts) else drafts
    replies = 0
    for d in sent.itertuples() if len(sent) else []:
        replies += bool(q("SELECT 1 FROM interactions WHERE lead_id = ? AND parent_responded = 1 AND kind = 'note' AND at > ?", (d.lead_id, d.sent_at)))
    done = acts[acts["status"].isin(["done", "superseded"])] if len(acts) else acts
    rows = [
        ("Follow-up discipline", "Active leads with Owner + Stage + NBA + Due", f"{int(complete.sum())}/{len(active)} ({complete.mean() * 100:.0f}%)" if len(active) else "-"),
        ("Follow-up discipline", "Open actions overdue", f"{len(overdue)}/{len(open_a)}"),
        ("Follow-up discipline", "Overdue > 1 operating day (escalated)", sum(1 for r in b["overdue"] if r["Escalate"])),
        ("Follow-up discipline", "Assessments / trials without outcomes", f"{len(b['assessments_no_outcome'])} / {len(b['trials_no_outcome'])}"),
        ("Follow-up discipline", "Actions completed", len(done)),
        ("CST feedback", "Next Best Actions overridden by CST", f"{int(acts['overridden'].sum()) if len(acts) else 0}/{len(acts)}"),
        ("Re-engagement", "Drafts created / sent / discarded", f"{len(drafts)} / {len(sent)} / {int((drafts['status'] == 'discarded').sum()) if len(drafts) else 0}"),
        ("Re-engagement", "Average CST edit rate on sent drafts", f"{sent['edit_ratio'].mean() * 100:.0f}%" if len(sent) else "-"),
        ("Re-engagement", "Parent reply rate after a sent draft", f"{replies}/{len(sent)}" if len(sent) else "-"),
    ]
    return quality.reset_index(), pd.DataFrame(rows, columns=["Area", "Measure", "Value"])

## 13. Evaluate the AI capture
A small labelled test set of realistic CST notes. Checks whether the model gets the level, subject, event and dates right (dates are relative to today so the test keeps working). Use it to compare models or prompt changes, and report the accuracy in the write-up.

In [ ]:
RUN_EVAL = True

def next_weekday(d, wd):
    return d + timedelta(days=(wd - d.weekday()) % 7 or 7)

def eval_cases():
    t = now().date()
    back = t + timedelta(days=10)
    tue = next_weekday(t, 1)
    return [
        (f"Mum interested in P4. Child weak in composition and comprehension. Looking for Saturday afternoon. Wants an assessment but travelling until {back:%d %B}. Asked us to WhatsApp after she returns.",
         {"level": "P4", "event": "assessment_interest", "contact_after": (back + timedelta(days=1)).isoformat()}),
        ("What do you have for my P4?", {"level": "P4", "event": "new_enquiry"}),
        ("Called Mrs Koh twice re Sec 2 Science, no answer, left voicemail.", {"level": "Sec 2", "subject": "science", "event": "no_answer"}),
        ("Jayden did the A Maths assessment today. Weak in algebra, OK in trig. Dad wants our recommendation.", {"event": "assessment_completed", "subject": "a maths"}),
        (f"Dad likes the trial but needs to discuss with wife, says fees a bit high. Call him next Tuesday.",
         {"event": "considering", "contact_after": tue.isoformat(), "decision_makers": True, "budget": True}),
        ("Mrs Ong says Sophie enjoyed the trial, wants to enrol in the Sunday P3 English class. Will pay this week.", {"level": "P3", "event": "enrolled"}),
        ("Parent replied no thanks, they found a home tutor.", {"event": "declined"}),
        ("Pri 6 girl, Science open-ended weak, PSLE next year. Mum wants to try a lesson first.", {"level": "P6", "subject": "science", "event": "trial_interest"}),
    ]

def run_eval():
    rows = []
    for note, exp in eval_cases():
        u = understand_note(note)
        f = u["fields"]
        checks = {}
        if "level" in exp: checks["level"] = f.get("level") == exp["level"]
        if "subject" in exp: checks["subject"] = any(exp["subject"] == s.lower() for s in (f.get("subjects") or []))
        if "event" in exp: checks["event"] = u["event"] == exp["event"]
        if "contact_after" in exp: checks["contact_after"] = f.get("contact_after") == exp["contact_after"]
        if "decision_makers" in exp: checks["decision_makers"] = bool(f.get("decision_makers"))
        if "budget" in exp: checks["budget"] = bool(f.get("budget_concern") or f.get("objections"))
        for k, ok in checks.items():
            got = {"level": f.get("level"), "subject": f.get("subjects"), "event": u["event"], "contact_after": f.get("contact_after"),
                   "decision_makers": f.get("decision_makers"), "budget": f.get("budget_concern")}[k]
            rows.append({"note": note[:60] + "...", "check": k, "expected": exp[k], "got": got, "pass": ok})
    df = pd.DataFrame(rows)
    print(f"Capture accuracy: {df['pass'].mean() * 100:.0f}% of {len(df)} checks")
    print(df.groupby("check")["pass"].mean().mul(100).round(0).astype(int).astype(str).add("%").to_string())
    return df

if RUN_EVAL:
    eval_df = run_eval()
    display(eval_df)

## 14. Walkthrough (the brief's own example)
Runs the full loop once on the example note from brief §5A, then prints the board and a draft. Good for checking everything works before launching the app. The walkthrough lead is removed afterwards unless `KEEP_WALKTHROUGH = True`.

In [ ]:
RUN_WALKTHROUGH = True
KEEP_WALKTHROUGH = False

def delete_lead(lead_id):
    for t in ("actions", "interactions", "drafts"):
        ex(f"DELETE FROM {t} WHERE lead_id = ?", (lead_id,))
    ex("DELETE FROM leads WHERE lead_id = ?", (lead_id,))
    persist_db()

if RUN_WALKTHROUGH:
    back = now().date() + timedelta(days=6)
    note = (f"Mum (Mrs Rachel Goh, 9123 4567) interested in P4 English for Lucas. Child weak in composition and comprehension. "
            f"Looking for Saturday afternoon. Wants an assessment but travelling until {back:%d %B}. Asked us to WhatsApp after she returns.")
    print("NOTE:", note, "\n")
    u = understand_note(note)
    print("1. UNDERSTAND - event:", u["event"], "| summary:", u["summary"])
    print(pd.DataFrame(review_rows(u), columns=["Field", "Value", "Check"]).query("Value != '' or Check != ''").to_string(index=False))
    print("   missing:", u["missing"], "| conflicts:", u["conflicts"], "| warnings:", u["warnings"])
    lead, ids = apply_capture(None, u["fields"], u["event"], note, "Clara", u["parent_responded"], u["raw"], summary=u["summary"])
    print(f"\n2. CAPTURE + NEXT BEST ACTION for {lead['lead_id']} (owner {lead['owner']}, stage {lead['stage']}):")
    for a in q(f"SELECT action_type, description, due_at, rule FROM actions WHERE id IN ({','.join('?' * len(ids))})", ids):
        print(f"   {a['action_type']} due {a['due_at']}\n   {a['description']}\n   rule: {a['rule']}")
    b = board()
    print("\n3. BOARD:", re.sub(r"\*\*", "", board_summary(b)))
    print("\n4. RE-ENGAGEMENT DRAFT for L007 (considering, cost concern, discussing with husband):")
    d = draft_message("L007")
    print(d.get("message") or d.get("blocked"))
    print("   sources:", d.get("sources"), "| guardrail warnings:", d.get("warnings"))
    if d.get("draft_id"):
        ex("UPDATE drafts SET status = 'discarded' WHERE id = ?", (d["draft_id"],))
    if not KEEP_WALKTHROUGH:
        delete_lead(lead["lead_id"])
        ex("DELETE FROM drafts WHERE id = ?", (d.get("draft_id", -1),))
        persist_db()

## 15. Launch the CST app
Open the `gradio.live` link printed below. It stays live while this cell runs.

| Tab | What the CST does |
|---|---|
| 1. Capture interaction | Paste a note after any parent interaction → review what the AI understood → confirm → see the Next Best Action and due date (override if you disagree) |
| 2. Today's board | Overdue (with escalations), due today, high-intent, assessments/trials without outcomes, leads without an NBA, stale leads |
| 3. Lead record | Full record, history, AI summary; record the outcome of an action; get an NBA for leads that have none |
| 4. Draft message | Context-aware follow-up/re-engagement draft from approved content; edit, send yourself, log it |
| 5. Success measures | Brief §14 measures, live |
| Action Standards | The rules the system applies, for transparency |

In [ ]:
import gradio as gr

EXAMPLE_NOTE = ("Mum interested in P4. Child weak in composition and comprehension. Looking for Saturday afternoon. "
                "Wants an assessment but travelling until 18 November. Asked us to WhatsApp after she returns.")

def lead_choices(active_only=False):
    rows = q("SELECT lead_id, parent_name, child_name, level, subjects, stage FROM leads ORDER BY lead_id")
    return [f"{r['lead_id']} · {r['parent_name'] or '?'} · {(r['level'] or '?')} {(r['subjects'] or '').replace(';', '/')} · {r['stage']}"
            for r in rows if not active_only or r["stage"] not in FINAL_STAGES]

def lid(choice):
    return choice.split(" · ")[0] if choice and re.match(r"^L\d+", choice) else None

def action_choices(lead_id):
    return [f"#{a['id']} · {a['action_type']} · due {a['due_at']}" for a in open_actions(lead_id)] if lead_id else []

def aid(choice):
    m = re.match(r"^#(\d+)", choice or "")
    return int(m.group(1)) if m else None

def nice_due(s):
    d = parse_dt(s)
    if not d:
        return "-"
    flag = " **(OVERDUE)**" if d < now() else ""
    return f"{DAYS[d.weekday()][:3]} {d:%d %b %H:%M}{flag}"

def lead_card(lead_id):
    l = get_lead(lead_id)
    acts = open_actions(lead_id)
    head = f"### {l['lead_id']} · {l['parent_name'] or '?'}{' (' + l['child_name'] + ')' if l['child_name'] else ''} · {l['level'] or '?'} {(l['subjects'] or '').replace(';', '/')}"
    need = l["learning_needs"] or l["parent_goals_concerns"] or "-"
    lines = [head, "", "| Owner | Stage | Parent need | Next Best Action | Due |", "|---|---|---|---|---|"]
    if acts:
        for a in acts:
            lines.append(f"| {a['owner']} | {l['stage']} | {need} | {a['description']} | {nice_due(a['due_at'])} |")
    else:
        lines.append(f"| {l['owner']} | {l['stage']} | {need} | {'(lead closed)' if l['stage'] in FINAL_STAGES else '**No Next Best Action**'} | - |")
    for a in acts:
        lines.append(f"\n*Rule applied ({a['action_type']}):* {a['rule']}" + (" *(overridden by CST)*" if a["overridden"] else ""))
    miss = missing_fields(l)
    if miss and l["stage"] not in FINAL_STAGES:
        lines.append(f"\n**Missing:** {', '.join(miss)}")
    if l["contact_after"]:
        lines.append(f"\n**Parent asked to be contacted on/after:** {l['contact_after']}")
    if l["closed_reason"]:
        lines.append(f"\n**Closed reason:** {l['closed_reason']}")
    return "\n".join(lines)

def profile_df(lead_id):
    l = get_lead(lead_id)
    keys = REVIEW_FIELDS + ["owner", "source", "nr_attempts", "last_parent_response_at", "created_at"]
    return pd.DataFrame([[k, l.get(k) if l.get(k) not in (None, "") else ""] for k in keys], columns=["Field", "Value"])

def history_df(lead_id):
    return pd.DataFrame(q("SELECT at AS 'When', author AS 'By', kind AS 'Type', event AS 'Event', raw_note AS 'Note' FROM interactions WHERE lead_id = ? ORDER BY at DESC", (lead_id,)))

def actions_df(lead_id):
    return pd.DataFrame(q("""SELECT id AS 'ID', action_type AS 'Action', due_at AS 'Due', status AS 'Status', outcome AS 'Outcome'
                             FROM actions WHERE lead_id = ? ORDER BY id DESC""", (lead_id,)))

def df_or_empty(rows, msg="Nothing here."):
    return pd.DataFrame(rows) if rows else pd.DataFrame([{"": msg}])

# ---------- tab 1 callbacks ----------
def ui_understand(lead_choice, note):
    if not (note or "").strip():
        return "Type or paste a note first.", None, None, True, {}
    u = understand_note(note.strip(), lid(lead_choice))
    md = [f"**AI understanding:** {u['summary']}", f"**Event:** `{u['event']}` → stage **{next_stage((u['existing'] or {}).get('stage'), u['event'])}**"]
    if u["conflicts"]:
        md.append("**Conflicts with the existing record:** " + "; ".join(u["conflicts"]))
    if u["missing"]:
        md.append("**Still missing (ask next time):** " + ", ".join(u["missing"]))
    for w in u["warnings"]:
        md.append(f"⚠️ {w}")
    md.append("Check the table, correct anything wrong, then press **Confirm and save**.")
    rows = pd.DataFrame(review_rows(u), columns=["Field", "Value", "Check"])
    return "\n\n".join(md), rows, u["event"], u["parent_responded"], u

def ui_save(state, table, event, responded, author):
    if not state:
        return "Press **Understand note** first.", gr.update(), gr.update(), {}
    fields = {}
    for r in table.itertuples(index=False):
        f, v = r[0], str(r[1] or "").strip()
        if f in LIST_FIELDS:
            fields[f] = [x.strip() for x in v.split(";") if x.strip()]
        else:
            fields[f] = v or None
    for f in ("assessment_at", "trial_at"):
        if fields.get(f) and not parse_dt(fields[f]):
            return f"⚠️ {f} must look like YYYY-MM-DD HH:MM.", gr.update(), gr.update(), state
    if fields.get("contact_after") and not parse_dt(fields["contact_after"]):
        return "⚠️ contact_after must look like YYYY-MM-DD.", gr.update(), gr.update(), state
    edited = sum(1 for k, v in fields.items() if (v or None) != (state["fields"].get(k) or None) and
                 not (isinstance(v, list) and v == (state["fields"].get(k) or [])))
    lead, ids = apply_capture(state["lead_id"], fields, event, state["note"], author, responded,
                              {**state["raw"], "cst_corrections": edited}, summary=state["summary"])
    md = f"✅ Saved. CST corrected {edited} field(s).\n\n" + lead_card(lead["lead_id"])
    choice = next(c for c in lead_choices() if c.startswith(lead["lead_id"]))
    return md, gr.update(choices=["New lead"] + lead_choices(), value=choice), gr.update(choices=action_choices(lead["lead_id"]), value=None), {}

def ui_override(action_choice, due, desc):
    if not aid(action_choice):
        return "Pick an action."
    if due and not parse_dt(due):
        return "Due must look like YYYY-MM-DD HH:MM."
    override_action(aid(action_choice), due or None, desc or None)
    a = q("SELECT lead_id FROM actions WHERE id = ?", (aid(action_choice),))[0]
    return "Override saved (counted as CST feedback).\n\n" + lead_card(a["lead_id"])

# ---------- tab 2 ----------
def ui_board(owner):
    b = board(owner)
    return (board_summary(b), df_or_empty(b["overdue"]), df_or_empty(b["due_today"]), df_or_empty(b["high_intent"]),
            df_or_empty(b["assessments_no_outcome"]), df_or_empty(b["trials_no_outcome"]), df_or_empty(b["no_nba"]),
            df_or_empty(b["stale"]), df_or_empty(b["upcoming"]))

# ---------- tab 3 ----------
def ui_open_lead(choice):
    l = lid(choice)
    if not l:
        return "Pick a lead.", None, None, None, gr.update(choices=[])
    refresh_time_based()
    return lead_card(l), profile_df(l), history_df(l), actions_df(l), gr.update(choices=action_choices(l), value=None)

def ui_summary(choice):
    return summarise_lead(lid(choice)) if lid(choice) else "Pick a lead."

def ui_recommend(choice):
    l = lid(choice)
    if not l:
        return ("Pick a lead.",) + (gr.update(),) * 4
    ids = recommend_for_lead(l)
    card, p, h, a, dd = ui_open_lead(choice)
    note = "Next Best Action added." if ids else "This lead already has an open action or is closed."
    return note + "\n\n" + card, p, h, a, dd

def ui_outcome(choice, action_choice, outcome, note, author):
    if not aid(action_choice) or not outcome:
        return ("Pick an action and an outcome.",) + (gr.update(),) * 4 + (gr.update(),)
    record_outcome(aid(action_choice), outcome, note, author)
    card, p, h, a, dd = ui_open_lead(choice)
    return ("Outcome recorded.\n\n" + card, p, h, a, dd, gr.update(choices=lead_choices(), value=next(c for c in lead_choices() if c.startswith(lid(choice)))))

# ---------- tab 4 ----------
def ui_draft_actions(choice):
    return gr.update(choices=action_choices(lid(choice)), value=(action_choices(lid(choice)) or [None])[0])

def ui_draft(choice, action_choice, author):
    if not lid(choice):
        return "", "Pick a lead.", {}
    d = draft_message(lid(choice), aid(action_choice), author)
    if d.get("blocked"):
        return "", f"⛔ {d['blocked']}", {}
    meta = [f"**Purpose:** {d['purpose']}", "**Approved sources used:** " + (", ".join(f"`{s}`" for s in d["sources"]) or "none")]
    meta += [f"⚠️ {w}" for w in d["warnings"]] or ["✅ Guardrail checks passed."]
    meta.append("Edit the draft as needed, send it from WhatsApp yourself, then press **I've sent this**.")
    return d["message"], "\n\n".join(meta), d

def ui_sent(state, text, author):
    if not state or not state.get("draft_id"):
        return "Draft a message first.", {}
    ratio, new_ids = mark_sent(state["draft_id"], text, author)
    nxt = q(f"SELECT action_type, due_at FROM actions WHERE id IN ({','.join('?' * len(new_ids))})", new_ids) if new_ids else []
    msg = f"Logged as sent. You changed {ratio * 100:.0f}% of the draft."
    if nxt:
        msg += f" Next action scheduled: **{nxt[0]['action_type']}** due {nice_due(nxt[0]['due_at'])} (if the parent replies, log it in tab 1)."
    return msg, {}

def ui_discard(state):
    if state and state.get("draft_id"):
        ex("UPDATE drafts SET status = 'discarded' WHERE id = ?", (state["draft_id"],))
        persist_db()
    return "Draft discarded (counted as CST feedback).", {}

# ---------- tab 5 ----------
def ui_metrics():
    quality, m = metrics()
    return quality, m

STANDARDS_MD = """
| Lead situation | LCentral Action Standard |
|---|---|
| New enquiry | Respond within 4 hours during operating hours. Outside operating hours: by the next operating day. |
| No response from parent | 1st follow-up within 1 day; 2nd follow-up 2-3 days later; final follow-up 5-7 days later. |
| Assessment interest / booking | Not booked: follow up within 1 day. Booked: confirm immediately; reminder 1 day before. |
| Assessment completed | Follow up same day with programme/class recommendation and next steps. |
| Trial booking | Confirm immediately; reminder 1 day before. |
| Trial completed | Follow up same day. |
| Parent requests later contact | Follow up on the agreed date; if no response, resume the sequence. |
| Parent considering | Follow up within 2-3 days; if still considering, again within 5-7 days. |
| Stale lead | 14 days without response after prescribed attempts (not if a future date was agreed). |
| Overdue action | Flag when overdue; escalate to the Centre Manager after more than 1 operating day. Assessments/trials first. |
| Non-responsive / declined | Stop active follow-up / close with reason. |

**Parent-specific commitments override the default cadence.** *Assumptions to confirm with LCentral:* trial interest (not booked) is followed up within 1 day; after two "considering" follow-ups the CST agrees a decision date.
"""

with gr.Blocks(title=f"{CENTRE_NAME} Lead Conversion Agent") as app:
    gr.Markdown(f"# {CENTRE_NAME} Lead Conversion Agent\n*The Agent captures, understands and recommends. The system reminds and highlights. The Centre Success Team decides and acts.*")
    author = gr.Dropdown(CST_TEAM, value=CST_TEAM[0], label="You are", scale=0)

    with gr.Tab("1. Capture interaction"):
        with gr.Row():
            c_lead = gr.Dropdown(["New lead"] + lead_choices(), value="New lead", label="Lead (pick existing or New lead)")
        c_note = gr.Textbox(lines=4, label="What happened? Type your note or paste the parent's message", placeholder=EXAMPLE_NOTE)
        c_btn = gr.Button("Understand note", variant="primary")
        c_md = gr.Markdown()
        c_table = gr.Dataframe(headers=["Field", "Value", "Check"], interactive=True, label="Review and correct (edit the Value column)", wrap=True)
        with gr.Row():
            c_event = gr.Dropdown([(f"{k}: {v}", k) for k, v in EVENTS.items()], label="What happened (event)")
            c_resp = gr.Checkbox(value=True, label="Parent responded in this interaction")
        c_save = gr.Button("Confirm and save", variant="primary")
        c_out = gr.Markdown()
        with gr.Accordion("Disagree with the recommendation? Override it", open=False):
            o_action = gr.Dropdown([], label="Action")
            o_due = gr.Textbox(label="New due (YYYY-MM-DD HH:MM)")
            o_desc = gr.Textbox(label="New description (optional)")
            o_btn = gr.Button("Save override")
        c_state = gr.State({})

    with gr.Tab("2. Today's board"):
        with gr.Row():
            b_owner = gr.Dropdown(["Everyone"] + CST_TEAM, value="Everyone", label="Show actions for")
            b_btn = gr.Button("Refresh", variant="primary")
        b_sum = gr.Markdown()
        gr.Markdown("### Overdue (escalated items first)")
        b_over = gr.Dataframe(wrap=True)
        gr.Markdown("### Due later today")
        b_today = gr.Dataframe(wrap=True)
        gr.Markdown("### High-intent leads needing attention (due within 24h or overdue)")
        b_high = gr.Dataframe(wrap=True)
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Assessments without outcomes")
                b_aout = gr.Dataframe(wrap=True)
            with gr.Column():
                gr.Markdown("### Trials without outcomes")
                b_tout = gr.Dataframe(wrap=True)
        gr.Markdown("### Active leads without a Next Best Action (open them in tab 3)")
        b_nonba = gr.Dataframe(wrap=True)
        gr.Markdown("### Stale leads (all attempts done, 14+ days silent)")
        b_stale = gr.Dataframe(wrap=True)
        gr.Markdown("### Coming up (next 7 days)")
        b_up = gr.Dataframe(wrap=True)

    with gr.Tab("3. Lead record"):
        with gr.Row():
            l_lead = gr.Dropdown(lead_choices(), label="Lead")
            l_open = gr.Button("Open", variant="primary")
        l_card = gr.Markdown()
        with gr.Row():
            l_sum_btn = gr.Button("AI summary")
            l_rec_btn = gr.Button("Recommend Next Best Action (for leads without one)")
        l_sum = gr.Markdown()
        gr.Markdown("### Record an outcome\nParent replied with new information? Log it in **tab 1** instead, so the AI can capture it.")
        with gr.Row():
            l_action = gr.Dropdown([], label="Open action")
            l_outcome = gr.Radio(OUTCOMES, label="Outcome")
        l_note = gr.Textbox(label="Note or reason (optional)")
        l_save = gr.Button("Save outcome")
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Lead record")
                l_prof = gr.Dataframe(wrap=True)
            with gr.Column():
                gr.Markdown("### Interaction history")
                l_hist = gr.Dataframe(wrap=True)
        gr.Markdown("### Actions")
        l_acts = gr.Dataframe(wrap=True)

    with gr.Tab("4. Draft message"):
        gr.Markdown("Action Standards decide **when** to follow up; the parent's context decides **what** to say. Drafts only use approved content and are never sent automatically.")
        with gr.Row():
            d_lead = gr.Dropdown(lead_choices(active_only=True), label="Lead")
            d_action = gr.Dropdown([], label="For action")
        d_btn = gr.Button("Draft message", variant="primary")
        d_text = gr.Textbox(lines=6, label="Draft (edit before sending)", interactive=True)
        d_meta = gr.Markdown()
        with gr.Row():
            d_sent = gr.Button("I've sent this", variant="primary")
            d_disc = gr.Button("Discard draft")
        d_msg = gr.Markdown()
        d_state = gr.State({})

    with gr.Tab("5. Success measures"):
        m_btn = gr.Button("Refresh", variant="primary")
        gr.Markdown("### Lead data quality: share of brief §1.2 fields captured")
        m_quality = gr.Dataframe()
        gr.Markdown("### Follow-up discipline, CST feedback and re-engagement")
        m_table = gr.Dataframe(wrap=True)

    with gr.Tab("Action Standards"):
        gr.Markdown(STANDARDS_MD)
        gr.Markdown("**Operating hours:** " + "; ".join(f"{DAYS[d]} {'closed' if not h else h[0] + '-' + h[1]}" for d, h in OPERATING_HOURS.items()) +
                    f"\n\n**CST team:** {', '.join(CST_TEAM)}. New leads go to whoever has the fewest open actions. Escalations go to the {CENTRE_MANAGER}.")

    c_btn.click(ui_understand, [c_lead, c_note], [c_md, c_table, c_event, c_resp, c_state])
    c_save.click(ui_save, [c_state, c_table, c_event, c_resp, author], [c_out, c_lead, o_action, c_state])
    o_btn.click(ui_override, [o_action, o_due, o_desc], [c_out])
    board_out = [b_sum, b_over, b_today, b_high, b_aout, b_tout, b_nonba, b_stale, b_up]
    b_btn.click(ui_board, [b_owner], board_out)
    b_owner.change(ui_board, [b_owner], board_out)
    l_open.click(ui_open_lead, [l_lead], [l_card, l_prof, l_hist, l_acts, l_action])
    l_lead.change(ui_open_lead, [l_lead], [l_card, l_prof, l_hist, l_acts, l_action])
    l_sum_btn.click(ui_summary, [l_lead], [l_sum])
    l_rec_btn.click(ui_recommend, [l_lead], [l_card, l_prof, l_hist, l_acts, l_action])
    l_save.click(ui_outcome, [l_lead, l_action, l_outcome, l_note, author], [l_card, l_prof, l_hist, l_acts, l_action, l_lead])
    d_lead.change(ui_draft_actions, [d_lead], [d_action])
    d_btn.click(ui_draft, [d_lead, d_action, author], [d_text, d_meta, d_state])
    d_sent.click(ui_sent, [d_state, d_text, author], [d_msg, d_state])
    d_disc.click(ui_discard, [d_state], [d_msg, d_state])
    m_btn.click(ui_metrics, [], [m_quality, m_table])
    app.load(ui_board, [b_owner], board_out)

app.launch(share=True, debug=True)